In [1]:
import anatomist.api as ana
from soma.qt_gui.qtThread import QtThreadCall
from soma.qt_gui.qt_backend import Qt

a = ana.Anatomist()

from soma import aims
import pandas as pd
import numpy as np
import os

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
existing QApplication: 0
QStandardPaths: XDG_RUNTIME_DIR not set, defaulting to '/tmp/runtime-ad279118'


create qapp
global modules: /casa/host/build/share/anatomist-5.2/python_plugins
home   modules: /casa/home/.anatomist/python_plugins
done
Starting Anatomist.....
config file : /casa/home/.anatomist/config/settings.cfg
PyAnatomist Module present
PythonLauncher::runModules()
loading module simple_controls
loading module save_resampled
loading module selection
loading module bsa_proba
loading module modelGraphs
loading module profilewindow
loading module ana_image_math
loading module paletteViewer
loading module foldsplit
loading module anacontrolmenu
loading module gradientpalette
loading module palettecontrols
loading module meshsplit
loading module volumepalettes
loading module gltf_io
loading module infowindow
loading module histogram
loading module statsplotwindow
loading module valuesplotwindow
all python modules loaded
Anatomist started.


#### To visualize specific 3D volumic sulci for specific subjects

In [56]:
dataset = 'UkBioBank40'
region = 'S.C.-sylv.' #"S.C.-sylv." "S.T.s."
side = 'L' #"L"

In [80]:
sorted_phenotype = pd.read_csv('/home/ad279118/tmp1/ses-2_T1_QC.csv')
sorted_phenotype = sorted_phenotype.sort_values(by='Inverted signal-to-noise ratio in T1', ascending=False)
sorted_phenotype.ID = sorted_phenotype['ID'].apply(lambda x : 'sub-'+str(x))
sample = sorted_phenotype['ID'].to_list()
sample = sample[-25:-10]
sorted_phenotype

,ID,Inverted contrast-to-noise ratio in T1,Inverted signal-to-noise ratio in T1
10455,sub-2232598,0.053741,0.027499
15615,sub-2829033,0.043208,0.026351
467,sub-1055585,0.049933,0.025946
5140,sub-1601614,0.059258,0.025870
22870,sub-3678218,0.046955,0.025684
...,...,...,...
22507,sub-3638454,0.026860,0.011719
30318,sub-4552612,0.029210,0.011677
38766,sub-5552533,0.028356,0.011412
35447,sub-5157126,0.026455,0.011255


In [79]:
outliers = pd.read_csv('/volatile/ad279118/2024_adufournet_sulcus_genetics/notebooks/UKB/QC/outliers.csv')
outliers

,ID
0,sub-2481728
1,sub-2632623
2,sub-3396940
3,sub-3716267
4,sub-5417598
...,...
98,sub-1467575
99,sub-5431753
100,sub-2715778
101,sub-5124493


In [92]:
base_model_path = '/neurospin/dico/data/deep_folding/current/models/Champollion_V0'
bdd_path = f'{base_model_path}/SC-sylv_right/11-43-38_3/ukb40_random_epoch100_embeddings/full_embeddings.csv'

bdd_ukb = pd.read_csv(bdd_path) 

merge = pd.merge(bdd_ukb[['ID', 'dim1']], outliers, on='ID')
merge

,ID,dim1
0,sub-1037379,-4.915963
1,sub-1232000,-6.713902
2,sub-1290934,3.846360
3,sub-1331493,-6.985346
4,sub-1467575,15.307404
...,...,...
92,sub-5866106,3.641518
93,sub-5890726,-36.871567
94,sub-5894417,-18.315098
95,sub-5946014,9.725495


In [65]:
volume = False
white_matter = True
nb_columns=7
block = a.createWindowsBlock(nb_columns) # nb of columns
dic_windows = {}

referential1 = a.createReferential()

mm_skeleton_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}crops'
dic_windows['Sulci_color']=a.loadObject('/casa/host/build/share/brainvisa-share-5.2/nomenclature/hierarchy/sulcal_root_colors.hie')

for i, subject_id in enumerate(sample):

    if volume:
        volume_path = f"{mm_skeleton_path}/{subject_id}_cropped_skeleton.nii.gz"
        
        if os.path.isfile(volume_path):
            vol = aims.read(volume_path)
            
            dic_windows[f'a_vol{subject_id}'] = a.toAObject(vol)
            dic_windows[f'rvol{subject_id}'] = a.fusionObjects(objects=[dic_windows[f'a_vol{subject_id}']], method='VolumeRenderingFusionMethod')
            dic_windows[f'rvol{subject_id}'].releaseAppRef()
            dic_windows[f'rvol{subject_id}'].assignReferential(referential1)

            dic_windows[f'wvr{subject_id}'] = a.createWindow('3D', block=block) #geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
            dic_windows[f'wvr{subject_id}'].addObjects(dic_windows[f'rvol{subject_id}'])
        else:
            print(f"{volume_path} is not a correct path, or the .nii.gz doesn't exist")

    path_to_t1mri = f'/home/ad279118/tmp/{subject_id}/ses-2/anat/t1mri/default_acquisition'

    if white_matter:
        white_matter_path = f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject_id}_{side}white.gii'
        if os.path.isfile(white_matter_path):
            # To visualize the white matter for specific people
            dic_windows[f'brain_{subject_id}'] = a.loadObject(white_matter_path)
            dic_windows[f'brain_{subject_id}'].assignReferential(referential1)
        else:
            print(f"{white_matter_path} is not a correct path, or the .white.gii doesn't exist")

    else:
        grey_matter_path = f'{path_to_t1mri}/default_analysis/segmentation/mesh/{subject_id}_{side}hemi.gii'
        if os.path.isfile(grey_matter_path):
            # To visualize the white matter for specific people
            dic_windows[f'brain_{subject_id}'] = a.loadObject(white_matter_path)
            dic_windows[f'brain_{subject_id}'].assignReferential(referential1)
        else:
            print(f"{grey_matter_path} is not a correct path, or the hemi.gii doesn't exist")
    
    found_labeld_sulci = False
    sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/{side}{subject_id}.arg'
    spam_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/spam_session_auto/{side}{subject_id}_spam_session_auto.arg'
    deep_labelled_sulci_path = f'{path_to_t1mri}/default_analysis/folds/3.1/deepcnn_session_auto/{side}{subject_id}_deepcnn_session_auto.arg'
    
    if os.path.isfile(spam_labelled_sulci_path):
        # To visualize the annotated sulci for specific people
        dic_windows[f'sulci_labelled_{subject_id}'] = a.loadObject(spam_labelled_sulci_path)
        dic_windows[f'sulci_labelled_{subject_id}'].assignReferential(referential1)
        found_labeld_sulci = True
    else:
        print(f"{spam_labelled_sulci_path} is not a correct path, or the .arg doesn't exist")
        print("Automatic try with 'deepcnn_session_auto' instead of 'spam_session_auto'")
        if  os.path.isfile(deep_labelled_sulci_path):
            # To visualize the annotated sulci for specific people
            dic_windows[f'sulci_labelled_{subject_id}'] = a.loadObject(deep_labelled_sulci_path)
            dic_windows[f'sulci_labelled_{subject_id}'].assignReferential(referential1)
            found_labeld_sulci = True

    if found_labeld_sulci:
        dic_windows[f'wws{subject_id}'] = a.createWindow('3D', block=block)
        dic_windows[f'wws{subject_id}'].addObjects([dic_windows[f'brain_{subject_id}'], dic_windows[f'sulci_labelled_{subject_id}']])

nifti transfo: 1
memory limit: 41470233804
Reading FGraph version 3.1
bounding box found : 75, 35, 38
                     155, 217, 183
nifti transfo: 2


ATransformSet::unregisterObserver: ref 0x5c66d60c3f30 not found


nifti transfo: 1
memory limit: 41478969753
Reading FGraph version 3.1
bounding box found : 67, 32, 55
                     163, 210, 206
nifti transfo: 2


ATransformSet::unregisterObserver: ref 0x5c66d60c3f30 not found


memory limit: 41476554752
Reading FGraph version 3.1
bounding box found : 84, 35, 40
                     158, 215, 155
nifti transfo: 1


...

ATransformSet::unregisterObserver: ref 0x5c66d60c3f30 not found


nifti transfo: 2
/home/ad279118/tmp/sub-2901062/ses-2/anat/t1mri/default_acquisition/default_analysis/folds/3.1/spam_session_auto/Lsub-2901062_spam_session_auto.arg is not a correct path, or the .arg doesn't exist
Automatic try with 'deepcnn_session_auto' instead of 'spam_session_auto'
memory limit: 41444150476
Reading FGraph version 3.3


bounding box found : 80, 28, 40
                     158, 207, 157
nifti transfo: 3


ATransformSet::unregisterObserver: ref 0x5c66d60c3f30 not found


Position : 129.691, 147.692, 111.291, 0
Position : 90.427, 165.309, 131.857, 0
no position could be read at 328, 362
Position : 103.058, 158.692, 114.144, 0
Position : 140.965, 107.69, 131.853, 0
Position : 119.283, 144.008, 62.7835, 0
no position could be read at 356, 336
no position could be read at 338, 309
no position could be read at 41, 235
Position : 132.167, 82.7884, 88.8075, 0
Position : 149.097, 116.078, 91.8241, 0
no position could be read at 51, 210
Position : 93.6885, 155.287, 125.733, 0
Position : 143.315, 102.409, 116.686, 0
Position : 99.0141, 125.034, 119.755, 0
Position : 139.865, 174.745, 109.711, 0
Position : 94.3231, 66.8783, 71.5333, 0
Position : 150.986, 112.652, 127.594, 0
Position : 136.357, 167.087, 112.659, 0
Position : 110.479, 56.4061, 75.0285, 0


#### To visualize the BUCKETS for specific people 

In [21]:
bucket_path = f'/neurospin/dico/data/deep_folding/current/datasets/{dataset}/crops/2mm/{region}/mask/{side}buckets'

bucket_files = []
bck_path = f'{bucket_path}/{subject_id}_cropped_skeleton.bck'

for subject_id in sample:
    if os.path. isfile(bck_path):
        bucket_files.append(bck_path)
    else:
        print(f"{bck_path} is not a correct path, or the .bck doesn't exist")

for i, file in enumerate(bucket_files):
    dic_windows[f'bck_{i}'] = a.loadObject(file)
    dic_windows[f'w_{i}'] = a.createWindow('3D', block=block)#geometry=[100+400*(i%3), 100+440*(i//3), 400, 400])
    dic_windows[f'w_{i}'].addObjects(dic_windows[f'bck_{i}'])

memory limit: 45595076198
Reading FGraph version 3.1


bounding box found : 16, 34, 49
                     88, 193, 156
nifti transfo: 1
nifti transfo: 1
memory limit: 45615969075
Reading FGraph version 3.1


bounding box found : 16, 36, 34
                     87, 208, 152
nifti transfo: 2
memory limit: 45611702681
Reading FGraph version 3.1


bounding box found : 16, 25, 35
                     95, 195, 150
nifti transfo: 1
nifti transfo: 1
memory limit: 45612046745
Reading FGraph version 3.1


bounding box found : 20, 33, 47
                     89, 204, 153
nifti transfo: 2
nifti transfo: 1
memory limit: 45606014156
Reading FGraph version 3.1


...

bounding box found : 19, 37, 39
                     95, 217, 150
nifti transfo: 2
nifti transfo: 2
memory limit: 45576975155
Reading FGraph version 3.1


bounding box found : 14, 21, 38
                     88, 176, 144
nifti transfo: 3
nifti transfo: 2
memory limit: 45596170649
Reading FGraph version 3.1


bounding box found : 15, 19, 42
                     88, 176, 150
nifti transfo: 3
nifti transfo: 1
memory limit: 45619894681
Reading FGraph version 3.1


bounding box found : 25, 35, 55
                     101, 202, 172
nifti transfo: 2


nifti transfo: 1
nifti transfo: 1
nifti transfo: 2
nifti transfo: 2
nifti transfo: 1
nifti transfo: 3
nifti transfo: 2
nifti transfo: 3
nifti transfo: 1
nifti transfo: 2
nifti transfo: 2
nifti transfo: 2
nifti transfo: 1
nifti transfo: 1
nifti transfo: 2
nifti transfo: 2
nifti transfo: 1
nifti transfo: 3
no position could be read at 221, 143
no position could be read at 197, 90
no position could be read at 209, 55
Position : -21.0162, -73.5132, -14.4223, 0
Position : -44.6912, -50.1271, -25.3672, 0
Position : -34.7306, 4.10713, -47.6484, 0
no position could be read at 173, 87
Position : 47.3809, 173.271, 52.9856, 0
Position : 42.7566, 142.728, 52.256, 0
no position could be read at 161, 136
Position : 38.034, 156.439, 47.4465, 0
Position : -61.719, 3.55056, -40.395, 0
Position : -41.341, -33.8254, -24.6045, 0
Position : -44.9775, 68.3375, -44.7286, 0
Position : -59.4141, -20.2527, -45.5596, 0
Position : -46.4447, 42.342, -77.61, 0
no position could be read at 181, 95
Position : 42.2152